# Version 2 : modèle supervisé calibré — inventaire de crues éclair méditerranéennes

Ce notebook construit et évalue la **Version 2** du cahier des charges (*Requirements Specification*, sections 6 et 8) :
un modèle supervisé (régression logistique et gradient boosting) **appris sur un inventaire d'inondations**
tiré d'événements documentés et d'**étiquettes satellites** (Copernicus EMS), avec des probabilités **calibrées**.

La Tunisie n'a qu'une seule cartographie satellite d'inondation (EMSR319, Nabeul, septembre 2018) : trop peu pour entraîner.
L'inventaire ajoute donc des **crues éclair cartographiées de la même façon sous un climat comparable** (Libye, Espagne, France, Grèce, Italie).
Chaque événement est testé **sans jamais avoir été vu à l'entraînement** ; le pli Nabeul est la validation tunisienne.

Métriques de la section 8 : **rappel au niveau Élevé et plus**, **fausses alertes**, **délai d'anticipation (lead time)**,
**calibration comparée à V1**, **utilité du facteur drainage**.

Scripts : `flood_risk/real/ems.py` (inventaire), `flood_risk/features.py`, `flood_risk/v1.py`, `flood_risk/v2.py`, `flood_risk/train.py`.

> **Note (version remplacée).** Ce notebook calcule la pluie à l'heure du pic de chaque orage, choisie après coup.
> L'audit qui a suivi (`pipeline_v2_temporel.ipynb`) reconstruit la pluie avec seulement l'information disponible à chaque instant.
> Une fois la pluie reconstruite ainsi, l'avantage de V2 complet sur le terrain seul n'est plus démontré.
>
> Le modèle retenu est donc le **modèle terrain** (`models/final/v2_terrain.joblib`) **avec un déclencheur pluie par zone**
> (`models/final/trigger.json`, provisoire). Voir `docs/MODEL_CONTRACT.md`.
>
> Les chiffres ci-dessous restent valables comme étude du classement pour un orage donné. Ils ne doivent pas être présentés
> comme la performance du modèle final, ni comme une prévision à +20 min.

In [1]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mp
from scipy.stats import mannwhitneyu
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

from flood_risk.schema import RAW_FEATURES, HISTORY_FEATURES, RELATIVE_FEATURES, DRAINAGE_FEATURES, LABEL, LEVELS
from flood_risk.features import engineer, MONOTONIC, drainage_vulnerability
from flood_risk.train import cross_validate, with_forecast_error, budget_recall, ece, NAMES, LEADS_MIN, MAIN_LEAD
from flood_risk.real.ems import EVENTS
from flood_risk import v1, v2

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
OUT, MODELS = Path("reports/med_imerg_tuned"), Path("models/med_imerg_tuned")
TUNING = Path("models/med_imerg/hgb_params")   # .json = meilleurs paramètres, .csv = toute la grille (flood_risk/tune.py)
PARAMS = json.loads(TUNING.with_suffix(".json").read_text())
ERA5_RESULTS = Path("reports/med/cv_metrics.csv")         # pluie ERA5, paramètres par défaut
IMERG_DEFAULT = Path("reports/med_imerg/cv_metrics.csv")  # pluie IMERG, paramètres par défaut
OUT.mkdir(parents=True, exist_ok=True); MODELS.mkdir(parents=True, exist_ok=True)
DATA = "data/inventory_med_imerg.csv.gz"
SEED, NEG_SAMPLE = 0, 0.3
LEVELS_MODE = "area"   # niveaux par budget d'alerte dans chaque zone (FR-24)
KINDS = ["v1", "v1_relative", "logistic", "hgb", "hgb_no_drainage", "hgb_terrain", "hgb_rain"]
COLORS = {"v1": "#94a3b8", "v1_relative": "#a16207", "logistic": "#2563eb", "hgb": "#dc2626", "hgb_no_drainage": "#f59e0b", "hgb_terrain": "#16a34a", "hgb_rain": "#7c3aed"}

def draw_flow(labels, title, figsize=(16, 3.2)):
    colors = ["#dbeafe", "#e0f2fe", "#dcfce7", "#fef9c3", "#fde68a", "#fbcfe8", "#e9d5ff", "#fee2e2"]
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off"); ax.set_xlim(0, len(labels) * 2); ax.set_ylim(0, 2)
    for i, lab in enumerate(labels):
        x = i * 2 + 0.1
        ax.add_patch(mp.FancyBboxPatch((x, 0.45), 1.7, 1.3, boxstyle="round,pad=0.02,rounding_size=0.12",
                                       fc=colors[i % len(colors)], ec="#334155"))
        ax.text(x + 0.85, 1.1, lab, ha="center", va="center", fontsize=9)
        if i < len(labels) - 1:
            ax.annotate("", xy=(x + 2.0, 1.1), xytext=(x + 1.75, 1.1),
                        arrowprops=dict(arrowstyle="-|>", lw=1.5, color="#334155"))
    ax.set_title(title, fontsize=12, weight="bold")
    plt.tight_layout(); plt.show()

draw_flow(["Cartes satellite\nCopernicus EMS\n(13 crues)", "Routes OSM\nsegments 150 m", "Terrain DEM 30 m\n+ béton\nWorldCover",
           "Pluie GPM IMERG\n10 km, 30 min", "Étiquette\ninondé = touche\nla zone d'eau", "Contrôle\nqualité",
           "Entraînement\nV2 + calibration", "Test : 1 crue\njamais vue\n(dont Nabeul)"],
          "Schéma du pipeline V2")

## 1. Sources et construction de l'inventaire

| Donnée | Source | Rôle |
|---|---|---|
| Zones inondées | Copernicus EMS Rapid Mapping (couche *observed event*, produits délimitation / gradation) | **étiquette** `flooded` |
| Emprise cartographiée | Copernicus EMS (*area of interest*) | limite de la zone étudiée |
| Routes, cours d'eau, passages | OpenStreetMap (Overpass) | segments de 150 m, distance à l'oued |
| Terrain | Copernicus GLO-30 (ouverture morphologique pour retirer les bâtiments) | altitude relative, pente, dépressions, TWI |
| Surfaces bâties | ESA WorldCover | imperméabilisation |
| Pluie | **NASA GPM IMERG V07** (satellites + pluviomètres, maille 0,1° ≈ 10 km, pas de 30 min), `flood_risk/real/imerg.py` | chaque segment prend la pluie de sa maille, à l'heure la plus pluvieuse de cette maille pendant l'orage |
| Pluie (ancienne version) | Open-Meteo, réanalyse ERA5 (≈ 30 km) | une seule valeur par zone ; gardée pour la comparaison de la section 7 bis |

**Règle d'étiquetage** : un segment est *inondé* s'il touche (tampon de 15 m) un polygone « zone inondée » ou « trace de crue ».
Une ligne = un segment × une crue. Les percentiles (`*_pct`) sont calculés **à l'intérieur de chaque zone** pour comparer des reliefs différents.

In [2]:
inv = pd.read_csv(DATA)
inv["event"] = inv.event_id.map(EVENTS)
print(f"lignes (segment × crue) : {len(inv):,}")
print(f"crues (événements)      : {inv.event_id.nunique()}")
print(f"zones cartographiées    : {inv.aoi.nunique()}")
print(f"segments inondés        : {inv[LABEL].sum():,} ({inv[LABEL].mean():.2%})")
per_event = inv.groupby(["event_id", "event"]).agg(zones=("aoi", "nunique"), segments=(LABEL, "size"),
                                                  inondes=(LABEL, "sum"), pluie_mm=("storm_total_mm", "max"),
                                                  pic_mmh=("rain_eta_mmh", "max"))
per_event["taux"] = (per_event.inondes / per_event.segments).map("{:.1%}".format)
per_event.round(1)

lignes (segment × crue) : 225,865
crues (événements)      : 13
zones cartographiées    : 28
segments inondés        : 24,565 (10.88%)


,,zones,segments,inondes,pluie_mm,pic_mmh,taux
event_id,event,,,,,,
EMSR257,"Mandra flash flood, Attica, Greece, Nov 2017",1,2777,481,138.3,25.4,17.3%
EMSR319,"Nabeul, Tunisia, Sept 2018",4,30718,184,78.2,17.1,0.6%
EMSR323,"Mallorca flash flood, Spain, Oct 2018",3,2558,648,85.3,16.5,25.3%
EMSR324,"Aude flash flood, France, Oct 2018",2,8157,250,88.2,20.1,3.1%
EMSR327,"Andalusia flash flood, Spain, Oct 2018",1,1272,29,29.6,5.4,2.3%
EMSR388,"South-east Spain DANA, Sept 2019",3,64831,7403,196.7,37.4,11.4%
EMSR413,"Western Attica, Greece, Nov 2019",2,7506,918,47.8,12.0,12.2%
EMSR480,"Crete, Greece, Nov 2020",2,5732,793,273.2,20.1,13.8%
EMSR548,"Eastern Sicily medicane, Italy, Oct 2021",2,35495,204,158.5,19.8,0.6%


**Analyse.** L'inventaire contient :

- **225 865 lignes** (segment × crue) ;
- **13 crues** et **28 zones cartographiées**, en Tunisie, Libye, Espagne, France, Grèce et Italie ;
- **24 565 segments inondés (10,9 %)**.

Deux activations n'ont pas donné de zone exploitable : Malaga 2018 et Crète 2026.
La colonne `pluie_mm` est maintenant le **cumul IMERG maximal** de la crue (maille la plus arrosée).

Trois points comptent pour la suite :

- **Le taux d'inondés varie énormément.** Il est de 0,6 % à Nabeul et en Sicile orientale, et de 20 à 27 % à Mallorca, en Libye et à Valence 2024.
  Copernicus trace ses zones autour des secteurs touchés : ce taux dépend donc **du découpage de la carte** autant que de la gravité de la crue.
- **IMERG corrige une partie des erreurs d'ERA5.** Mallorca passe de 15 à 85 mm, Crète 2020 de 58 à 273 mm, Mandra de 38 à 138 mm.
  À Nabeul, IMERG reste faible (78 mm au maximum contre plus de 200 mm mesurés localement le 22 septembre 2018) : une maille de 10 km lisse encore les cellules orageuses les plus intenses.
- Treize crues, c'est encore peu pour un modèle supervisé. Chaque résultat « par crue » repose sur un seul événement et doit être lu comme une indication, pas comme une preuve.

## 2. Contrôle qualité (QC)

In [3]:
F = RAW_FEATURES + RELATIVE_FEATURES
qc = pd.DataFrame({"manquantes": inv[F].isna().sum(), "min": inv[F].min(), "médiane": inv[F].median(),
                   "max": inv[F].max(), "distinctes": inv[F].nunique()})
qc["constante"] = qc.distinctes <= 1
qc.round(2)

,manquantes,min,médiane,max,distinctes,constante
elevation_rel_m,0,0.00,8.87,336.30,179700,False
slope_pct,0,0.00,2.12,165.32,187432,False
twi,0,3.44,8.50,18.73,187602,False
depression_depth_m,0,0.00,0.00,40.86,20727,False
dist_wadi_m,0,0.00,398.00,5000.00,212332,False
impervious_frac,0,0.00,0.46,1.00,590,False
drain_inlet_density,0,0.00,0.00,60.00,38,False
osm_completeness,0,0.50,0.50,0.50,1,True
underpass,0,0.00,0.00,1.00,2,False
ford,0,0.00,0.00,1.00,2,False


In [4]:
# Les variables de pluie sont-elles constantes à l'intérieur d'une zone ?
rain_cols = ["rain_eta_mmh", "rain_1h_mm", "rain_3h_mm", "rain_24h_mm", "dry_spell_days"]
print("valeurs distinctes de pluie par zone (max) :", inv.groupby("aoi")[rain_cols].nunique().max().to_dict())
dup = inv.duplicated(["segment_id", "event_id"]).sum()
print("doublons segment × crue :", dup, "(zones qui se chevauchent dans une même crue)")
# Un segment vu par deux zones d'une même crue est gardé une fois, inondé si l'une des cartes le dit.
inv = inv.sort_values(LABEL).drop_duplicates(["segment_id", "event_id"], keep="last").sort_index()
print(f"après dédoublonnage : {len(inv):,} lignes, {inv[LABEL].sum():,} inondées ({inv[LABEL].mean():.2%})")

valeurs distinctes de pluie par zone (max) : {'rain_eta_mmh': 23, 'rain_1h_mm': 24, 'rain_3h_mm': 24, 'rain_24h_mm': 24, 'dry_spell_days': 1}
doublons segment × crue : 9694 (zones qui se chevauchent dans une même crue)


après dédoublonnage : 216,171 lignes, 23,340 inondées (10.80%)


In [5]:
by_aoi = inv.groupby("aoi").agg(segments=(LABEL, "size"), taux=(LABEL, "mean"), event=("event_id", "first")).sort_values("taux")
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(by_aoi.index, 100 * by_aoi.taux, color=["#dc2626" if e == "EMSR319" else "#64748b" for e in by_aoi.event])
ax.set_yscale("log"); ax.set_ylabel("segments inondés (%) — échelle log")
ax.set_title("Densité d'étiquettes par zone (rouge = Nabeul)")
plt.xticks(rotation=70, fontsize=8); plt.tight_layout(); plt.show()

**Analyse du contrôle qualité.**

- **Aucune valeur manquante.**
- **Variables constantes, donc inutiles ici :**
  - `osm_completeness` est fixée à 0,5 hors du pilote ;
  - `black_spot_count` est nulle partout, faute d'historique de points noirs.
- **`drain_inlet_density` n'a que 38 valeurs distinctes** : OpenStreetMap ne cartographie presque aucun avaloir autour de la Méditerranée.
- **La pluie varie maintenant à l'intérieur des zones.**
  - IMERG donne jusqu'à **23–24 valeurs distinctes par zone** (6 mailles en médiane), contre une seule avec ERA5.
  - Le pic de pluie va de **3,9 à 37,6 mm/h** selon le segment.
  - Seul `dry_spell_days` (jours secs avant l'orage) garde une valeur par zone, car il vient encore de la réanalyse.
- **9 694 doublons** viennent de zones d'une même crue qui se chevauchent.
  - Ils sont fusionnés : un segment est inondé si une des cartes le dit.
  - Il reste **216 171 lignes, dont 23 340 inondées (10,8 %)**.
- **La densité d'étiquettes va de 0 % à plus de 40 % selon la zone** (graphique en échelle logarithmique).
  - Les modèles doivent donc être jugés **crue par crue** (ROC-AUC, rappel au budget), plutôt que sur des probabilités absolues.

## 3. Exploration : qu'est-ce qui distingue un segment inondé ?

In [6]:
cols = ["elevation_rel_m", "elevation_rel_pct", "dist_wadi_m", "dist_wadi_pct", "slope_pct", "twi", "twi_pct",
        "depression_depth_m", "impervious_frac"]
s = inv.groupby(LABEL)[cols].median().T
s.columns = ["médiane non inondé", "médiane inondé"]
s["ROC-AUC seule (1 = parfait)"] = [max(roc_auc_score(inv[LABEL], inv[c]), 1 - roc_auc_score(inv[LABEL], inv[c])) for c in cols]
s.round(3)

,médiane non inondé,médiane inondé,ROC-AUC seule (1 = parfait)
elevation_rel_m,9.942,5.161,0.679
elevation_rel_pct,0.529,0.301,0.665
dist_wadi_m,426.759,273.929,0.586
dist_wadi_pct,0.510,0.430,0.563
slope_pct,2.294,1.175,0.617
twi,8.427,9.002,0.588
twi_pct,0.486,0.588,0.570
depression_depth_m,0.000,0.000,0.566
impervious_frac,0.478,0.435,0.524


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, ["elevation_rel_pct", "dist_wadi_pct", "twi_pct"]):
    data = [inv.loc[(inv.event_id == e) & (inv[LABEL] == k), c] for e in per_event.index.get_level_values(0) for k in (0, 1)]
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, widths=0.6)
    for i, b in enumerate(bp["boxes"]):
        b.set_facecolor("#cbd5e1" if i % 2 == 0 else "#f87171")
    ax.set_xticks(np.arange(1.5, 2 * len(per_event), 2), per_event.index.get_level_values(0), rotation=60, fontsize=8)
    ax.set_title(f"{c} (gris = sec, rouge = inondé)")
plt.tight_layout(); plt.show()

**Analyse.** Comparés aux segments secs, les segments inondés sont :

- **plus bas dans leur zone** (altitude relative médiane 5,2 m contre 9,9 m, percentile 0,30 contre 0,53) ;
- **plus près d'un cours d'eau** (274 m contre 427 m) ;
- **plus plats** (pente 1,2 % contre 2,3 %) ;
- **plus humides topographiquement** (TWI plus élevé).

C'est le mécanisme d'une crue éclair de plaine : l'eau s'étale dans les points bas proches de l'oued.

Prise seule, la meilleure variable est l'**altitude relative** (ROC-AUC 0,68). Viennent ensuite son percentile (0,67) et la pente (0,62).

Les boîtes par crue montrent que ce signal existe dans la plupart des crues, avec deux exceptions nettes : **Mandra (Grèce)** et **Derna (Libye)**.
Dans ces deux cas, un torrent encaissé ou une rupture de barrage a traversé la ville, et l'eau a touché des rues à toutes les altitudes.

Le béton (`impervious_frac`) est **un peu plus faible** chez les inondés (0,44 contre 0,48). En crue éclair, ce sont surtout les terres basses périurbaines qui débordent, pas les centres-villes. C'est l'inverse de l'inondation urbaine de New York.

## 4. Découpage train / test : une crue entière à la fois

Une même crue ne doit **jamais** être à la fois dans l'entraînement et dans le test, sinon le modèle « reconnaît » la zone
et les scores sont gonflés. On utilise donc une validation croisée **leave-one-event-out** : un pli par crue, chacun teste une crue
entière (toutes ses zones) avec un modèle appris sur toutes les autres.

À l'intérieur de chaque pli d'entraînement, la **calibration** et les **seuils de niveaux** sont appris sur des prédictions
hors-pli (elles aussi groupées par crue), donc la crue testée n'influence rien.

In [8]:
events = list(per_event.index.get_level_values(0))
fig, ax = plt.subplots(figsize=(10, 3.6))
for i, test in enumerate(events):
    for j, e in enumerate(events):
        ax.add_patch(plt.Rectangle((j, len(events) - 1 - i), 0.95, 0.9, color="#dc2626" if e == test else "#93c5fd"))
ax.set_xlim(0, len(events)); ax.set_ylim(0, len(events))
ax.set_xticks(np.arange(len(events)) + 0.47, events, rotation=45, fontsize=8)
ax.set_yticks(np.arange(len(events)) + 0.45, [f"pli {len(events) - k}" for k in range(len(events))], fontsize=8)
ax.set_title("Leave-one-event-out : rouge = crue de test, bleu = entraînement")
plt.tight_layout(); plt.show()

## 5. Variables (features)

`engineer()` transforme les colonnes brutes en variables du modèle. Les contraintes **monotones** imposent au gradient boosting
un sens physique (plus de pluie ne peut pas baisser le risque, plus haut au-dessus du fond de vallée ne peut pas l'augmenter).

In [9]:
X_all = engineer(inv)
desc = {
    "elevation_rel_m": "altitude au-dessus du minimum local", "twi": "indice d'humidité topographique",
    "depression_depth_m": "profondeur de cuvette", "impervious_frac": "part de béton (100 m)",
    "drain_inlet_density": "avaloirs OSM / km", "osm_completeness": "qualité OSM (fixée)",
    "underpass": "passage inférieur", "ford": "gué", "black_spot_count": "points noirs connus",
    "rain_eta_mmh": "pluie prévue (mm/h)", "rain_1h_mm": "pluie 1 h", "rain_3h_mm": "pluie 3 h",
    "rain_24h_mm": "pluie 24 h", "dry_spell_days": "jours secs avant", "flood_reports_250m": "signalements < 250 m",
    "flood_reports_365d": "signalements 365 j", "basin_reports_prior": "avaloirs bouchés signalés",
    "elevation_rel_pct": "percentile d'altitude dans la zone", "dist_wadi_pct": "percentile de distance à l'oued",
    "twi_pct": "percentile TWI", "impervious_pct": "percentile béton", "drainage_vuln": "vulnérabilité drainage (FR-08)",
    "rain_excess_ratio": "pluie / seuil absorbé par le drainage", "runoff_index": "ruissellement = pluie × béton",
    "wadi_rain": "pluie × proximité oued", "log_dist_wadi": "log distance oued", "log_slope": "log pente"}
pd.DataFrame({"description": [desc.get(c, "") for c in X_all.columns],
              "monotone": [{1: "↑ risque", -1: "↓ risque", 0: "libre"}[MONOTONIC.get(c, 0)] for c in X_all.columns],
              "distinctes": X_all.nunique().values,
              "groupe drainage": [c in DRAINAGE_FEATURES or c in ("drainage_vuln", "rain_excess_ratio") for c in X_all.columns]},
             index=X_all.columns)

,description,monotone,distinctes,groupe drainage
elevation_rel_m,altitude au-dessus du minimum local,↓ risque,172213,False
twi,indice d'humidité topographique,↑ risque,178059,False
depression_depth_m,profondeur de cuvette,↑ risque,19588,False
impervious_frac,part de béton (100 m),↑ risque,589,False
drain_inlet_density,avaloirs OSM / km,↓ risque,35,True
osm_completeness,qualité OSM (fixée),libre,1,True
underpass,passage inférieur,↑ risque,2,False
ford,gué,↑ risque,2,False
black_spot_count,points noirs connus,↑ risque,1,False
rain_eta_mmh,pluie prévue (mm/h),↑ risque,155,False


**Analyse.** Le modèle reçoit **27 variables**, mais certaines n'apportent rien dans cet inventaire :

- `osm_completeness`, `black_spot_count` et les trois variables d'historique sont constantes ;
- `underpass` et `ford` sont très rares.

Avec IMERG, les variables de pluie (pic, cumuls sur 1 h, 3 h et 24 h, et leurs dérivées) **varient d'un segment à l'autre** : elles prennent environ 150 valeurs distinctes.
Le modèle combine donc deux types de signal :

- le **terrain** (où l'eau s'accumule) ;
- la **pluie locale** (combien il a plu sur cette maille).

Les contraintes monotones empêchent le modèle d'apprendre des relations absurdes, par exemple « plus de pluie, moins de risque ».

## 6. Entraînement et test

Modèles comparés :

| Code | Modèle |
|---|---|
| `v1` | V1 : indice multicritère à poids fixes (référence du cahier des charges) |
| `v1_relative` | règle simple : bas dans la zone + proche de l'oued (percentiles) |
| `logistic` | V2 régression logistique calibrée |
| `hgb` | V2 gradient boosting monotone calibré |
| `hgb_no_drainage` | V2 gradient boosting **sans** les variables drainage (ablation, section 8) |
| `hgb_terrain` | V2 gradient boosting **sans les variables de pluie** : apprend seulement *où* l'eau s'étale dans une zone |
| `hgb_rain` | V2 gradient boosting **avec la pluie seulement** (pic, cumuls 1 h, 3 h, 24 h, jours secs) : référence « pluie seule » |

Pourquoi `hgb_terrain` ? Avec la réanalyse ERA5, la pluie était **la même pour tous les segments d'une zone** et le modèle
apprenait mieux sans elle. Avec IMERG, la pluie varie d'une maille de 10 km à l'autre : comparer `hgb` et `hgb_terrain`
mesure maintenant ce que la pluie satellite apporte.

Pour chaque pli : on garde tous les segments inondés et 30 % des secs (correction de prior King & Zeng ensuite),
la pluie d'entraînement reçoit une erreur de prévision d'un délai aléatoire 0–30 min, puis on teste la crue exclue
à des délais de **0, 10, 20 et 30 min**.

### 6.1 Réglage des hyperparamètres du gradient boosting

`flood_risk/tune.py` essaie 16 combinaisons (taux d'apprentissage, taille des arbres, taille minimale des feuilles, part des variables
vue par chaque arbre). Chaque combinaison est jugée par son PR-AUC moyen sur les 13 crues, chaque crue étant prédite par un modèle
qui ne l'a pas vue. Les meilleurs paramètres sont ensuite utilisés pour tous les modèles de gradient boosting ci-dessous.

In [10]:
grid = pd.read_csv(TUNING.with_suffix(".csv"))
print("paramètres retenus :", PARAMS, "| par défaut :", {k: v2.HGB_DEFAULTS.get(k) for k in PARAMS})
grid.round(4)

paramètres retenus : {'learning_rate': 0.03, 'max_leaf_nodes': 15, 'min_samples_leaf': 40, 'max_features': 0.5} | par défaut : {'learning_rate': 0.05, 'max_leaf_nodes': 15, 'min_samples_leaf': 40, 'max_features': None}


,learning_rate,max_leaf_nodes,min_samples_leaf,max_features,pr_auc,pr_auc_std,recall_top2
0,0.03,15,40,0.5,0.2387,0.1211,0.1130
1,0.08,15,40,0.5,0.2385,0.1209,0.1126
2,0.08,31,40,0.5,0.2385,0.1192,0.1099
3,0.03,15,40,1.0,0.2376,0.1199,0.1105
4,0.08,15,40,1.0,0.2372,0.1226,0.1060
5,0.03,31,40,0.5,0.2363,0.1221,0.1106
6,0.03,31,40,1.0,0.2361,0.1188,0.1082
7,0.08,31,40,1.0,0.2343,0.1212,0.1050
8,0.03,15,200,0.5,0.2343,0.1277,0.1094
9,0.03,31,200,1.0,0.2338,0.1269,0.1039


**Analyse du réglage.** Les 16 combinaisons donnent toutes un PR-AUC entre **0,229 et 0,239**. L'écart entre la meilleure et la pire (0,010) est **dix fois plus petit que l'écart d'une crue à l'autre** (écart-type de 0,12).

Le modèle est donc **peu sensible à ses hyperparamètres**. Ce sont les données qui le limitent : nombre de crues, résolution de la pluie, étiquettes satellites.

Deux tendances se dégagent quand même :

- les **feuilles de 40 segments** sont meilleures que celles de 200 ;
- **regarder la moitié des variables par arbre** (`max_features` = 0,5) aide légèrement, car les arbres deviennent moins dépendants du béton.

Paramètres retenus : `learning_rate` 0,03, `max_leaf_nodes` 15, `min_samples_leaf` 40, `max_features` 0,5.

Le choix s'est fait sur la moyenne des 13 crues, dont Nabeul (1/13 du critère). Les scores de Nabeul après réglage sont donc **très légèrement optimistes**.

In [11]:
COLS = RAW_FEATURES + HISTORY_FEATURES + RELATIVE_FEATURES + [LABEL, "event_id", "segment_id", "aoi", "rain_interpolated"]
df = inv[[c for c in COLS if c in inv.columns]].copy()
floats = df.select_dtypes("float64").columns
df[floats] = df[floats].astype("float32")
res, oof = cross_validate(df, n_folds=df.event_id.nunique(), seed=SEED, neg_sample=NEG_SAMPLE, kinds=KINDS, levels=LEVELS_MODE, params=PARAMS)
res.to_csv(OUT / "cv_metrics.csv", index=False)
res.shape

fold 1/13 done (1 test events)


fold 2/13 done (1 test events)


fold 3/13 done (1 test events)


fold 4/13 done (1 test events)


fold 5/13 done (1 test events)


fold 6/13 done (1 test events)


... (4 outputs left out)

fold 11/13 done (1 test events)


fold 12/13 done (1 test events)


fold 13/13 done (1 test events)


(728, 19)

## 7. Résultats globaux (crues jamais vues, délai 20 min)

In [12]:
main = res[(res.target == "flooded") & (res.lead == MAIN_LEAD)]
cols = ["roc_auc", "pr_auc", "recall_top1", "recall_top2", "recall_top5", "recall_high", "precision_high", "accuracy_high"]
summary = main.groupby("model")[cols].mean().loc[KINDS]
summary.index = [NAMES[k] for k in KINDS]
summary.round(3)

,roc_auc,pr_auc,recall_top1,recall_top2,recall_top5,recall_high,precision_high,accuracy_high
V1 multi-criteria,0.577,0.144,0.015,0.027,0.067,0.029,0.173,0.875
Relative susceptibility,0.743,0.223,0.050,0.078,0.198,0.081,0.233,0.878
V2 logistic,0.694,0.233,0.037,0.072,0.160,0.073,0.257,0.879
V2 gradient boosting,0.739,0.208,0.041,0.084,0.187,0.101,0.264,0.879
V2 GB without drainage proxies,0.732,0.205,0.053,0.089,0.185,0.099,0.272,0.879
V2 GB terrain only,0.738,0.242,0.060,0.100,0.216,0.108,0.282,0.880
V2 GB rain only,0.499,0.130,0.012,0.029,0.063,0.027,0.125,0.873


In [13]:
pivot = main.pivot_table(index="events", columns="model", values="roc_auc").loc[events, KINDS]
pivot_pr = main.pivot_table(index="events", columns="model", values="pr_auc").loc[events, KINDS]
base = inv.groupby("event_id")[LABEL].mean().loc[events]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, tab, title in [(axes[0], pivot, "ROC-AUC par crue testée"), (axes[1], pivot_pr.div(base, axis=0), "PR-AUC / taux de base (gain)")]:
    im = ax.imshow(tab.values, cmap="RdYlGn", aspect="auto", vmin=0.3 if "ROC" in title else 0, vmax=1 if "ROC" in title else 4)
    ax.set_xticks(range(len(KINDS)), [NAMES[k] for k in KINDS], rotation=25, fontsize=8, ha="right")
    ax.set_yticks(range(len(events)), [f"{e} {EVENTS[e][:22]}" for e in events], fontsize=8)
    for i in range(tab.shape[0]):
        for j in range(tab.shape[1]):
            ax.text(j, i, f"{tab.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(title); plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

**Analyse des résultats globaux.** Moyenne sur les 13 crues, chacune testée sans avoir été vue, avec un délai de 20 min et les paramètres réglés :

| Modèle | ROC-AUC | PR-AUC | Rappel top 5 % |
|---|---|---|---|
| V1 multicritère | 0,58 | 0,14 | 0,07 |
| Règle relative (bas + proche oued) | **0,74** | 0,22 | 0,20 |
| V2 logistique | 0,69 | 0,23 | 0,16 |
| **V2 gradient boosting (terrain + pluie)** | **0,74** | 0,21 | 0,19 |
| V2 GB sans drainage | 0,73 | 0,21 | 0,19 |
| V2 GB terrain seul | **0,74** | **0,24** | **0,22** |
| V2 GB pluie seule | 0,50 | 0,13 | 0,06 |

1. **V2 bat nettement V1** : 0,74 contre 0,58 en ROC-AUC. V1, avec ses poids et seuils fixes, ne se transfère pas d'une région à l'autre.
2. **Le réglage apporte un petit gain** à V2 gradient boosting : ROC-AUC 0,736 → 0,739, rappel top 5 % 17,7 → 18,7 %.
3. **En moyenne, les trois meilleures approches se valent** : V2 complet, V2 terrain et la règle relative.
   - V2 terrain garde le meilleur PR-AUC et le meilleur rappel au budget.
   - V2 complet est meilleur sur les crues les plus étendues, où la pluie varie le plus d'une maille à l'autre :
     - SE Espagne : 0,77 contre 0,70 ;
     - Valence 2024 : 0,73 contre 0,67 ;
     - **Nabeul : 0,81 contre 0,80**.
4. **Les résultats varient beaucoup d'une crue à l'autre** (carte de chaleur) : V2 va de 0,32 (Mandra) à 0,93 (Sicile 2021).
   - **Mandra et Derna (Libye)** ne sont prévues par aucun modèle (ROC-AUC ≤ 0,57). Ce sont des crues torrentielles urbaines, pas des débordements de plaine.
   - Pour les prévoir, il faudrait d'autres variables : surface du bassin versant en amont, torrent canalisé sous la ville.
5. **La pluie seule ne classe pas les segments** (ROC-AUC 0,50, soit le hasard). Elle ne sait pas *où* l'eau va s'accumuler dans une zone.
   C'est son **association avec le terrain** qui aide V2 sur les crues étendues : la même averse ne déborde pas de la même façon dans un point bas et sur une colline (section 7 ter).

## 7 bis. Pluie ERA5 contre pluie IMERG

Même protocole (13 crues, leave-one-event-out, délai 20 min, niveaux par budget) et mêmes paramètres par défaut : seule la source de pluie change.
(Les deux runs ont été faits avant le réglage des hyperparamètres, pour isoler l'effet de la pluie.)
Les modèles qui n'utilisent pas la pluie (`v1_relative`, `hgb_terrain`) servent de témoin : leur score doit rester identique.

In [14]:
era5 = pd.read_csv(ERA5_RESULTS)
imerg0 = pd.read_csv(IMERG_DEFAULT)
imerg0 = imerg0[(imerg0.target == "flooded") & (imerg0.lead == MAIN_LEAD)]
era5 = era5[(era5.target == "flooded") & (era5.lead == MAIN_LEAD)]
CK = [k for k in KINDS if k != "hgb_rain"]   # les runs ERA5 / IMERG par défaut n'avaient pas la référence pluie seule
comp_cols = ["roc_auc", "pr_auc", "recall_top5", "precision_high"]
comp = pd.concat({"ERA5": era5.groupby("model")[comp_cols].mean(), "IMERG": imerg0.groupby("model")[comp_cols].mean()}, axis=1).loc[CK]
comp.index = [NAMES[k] for k in CK]
display(comp.round(3))
nab_comp = pd.concat({"ERA5": era5[era5.events == "EMSR319"].set_index("model")[comp_cols],
                      "IMERG": imerg0[imerg0.events == "EMSR319"].set_index("model")[comp_cols]}, axis=1).loc[CK]
nab_comp.index = [NAMES[k] for k in CK]
print("Nabeul 2018 :"); display(nab_comp.round(3))
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(CK))
ax.bar(x - 0.2, comp[("ERA5", "roc_auc")], 0.4, color="#94a3b8", label="pluie ERA5 (≈ 30 km)")
ax.bar(x + 0.2, comp[("IMERG", "roc_auc")], 0.4, color="#2563eb", label="pluie IMERG (≈ 10 km)")
ax.set_xticks(x, comp.index, rotation=20, ha="right", fontsize=8); ax.set_ylim(0.4, 0.85)
ax.axhline(0.5, color="k", ls=":", lw=1); ax.set(ylabel="ROC-AUC moyenne (13 crues)", title="Apport de la pluie satellite IMERG")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

ERA5                                    \
                               roc_auc pr_auc recall_top5 precision_high   
V1 multi-criteria                0.503  0.117       0.046          0.147   
Relative susceptibility          0.743  0.223       0.198          0.233   
V2 logistic                      0.614  0.165       0.162          0.199   
V2 gradient boosting             0.687  0.199       0.187          0.257   
V2 GB without drainage proxies   0.696  0.201       0.199          0.266   
V2 GB terrain only               0.733  0.239       0.213          0.285   

                                 IMERG                                    
                               roc_auc pr_auc recall_top5 precision_high  
V1 multi-criteria                0.577  0.144       0.067          0.173  
Relative susceptibility          0.743  0.223       0.198          0.233  
V2 logistic                      0.694  0.233       0.160          0.257  
V2 gradient boosting             0.736  0.205       0.177          0.268  
V2 GB without drainage proxies   0.729  0.200       0.168          0.272  
V2 GB terrain only               0.733  0.239       0.213          0.285

Nabeul 2018 :


ERA5                                    \
                               roc_auc pr_auc recall_top5 precision_high   
V1 multi-criteria                0.516  0.006       0.038          0.007   
Relative susceptibility          0.802  0.030       0.348          0.042   
V2 logistic                      0.663  0.013       0.158          0.038   
V2 gradient boosting             0.739  0.024       0.147          0.078   
V2 GB without drainage proxies   0.729  0.021       0.141          0.069   
V2 GB terrain only               0.774  0.015       0.130          0.064   

                                 IMERG                                    
                               roc_auc pr_auc recall_top5 precision_high  
V1 multi-criteria                0.546  0.007       0.060          0.011  
Relative susceptibility          0.802  0.030       0.348          0.042  
V2 logistic                      0.695  0.011       0.060          0.036  
V2 gradient boosting             0.798  0.018       0.201          0.098  
V2 GB without drainage proxies   0.765  0.017       0.168          0.090  
V2 GB terrain only               0.774  0.015       0.130          0.064

In [15]:
# La pluie IMERG explique-t-elle mieux la gravité des crues (taux d'inondés par zone) ?
from scipy.stats import spearmanr
z = inv.groupby("aoi").agg(taux=(LABEL, "mean"), pic=("rain_eta_mmh", "mean"), total=("storm_total_mm", "mean"),
                           mailles=("imerg_cell", "nunique"))
print(f"mailles IMERG par zone : médiane {z.mailles.median():.0f}, max {z.mailles.max()}")
print(f"Spearman taux d'inondés / pic IMERG   : {spearmanr(z.taux, z.pic)[0]:.2f}")
print(f"Spearman taux d'inondés / total IMERG : {spearmanr(z.taux, z.total)[0]:.2f}")
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(z.pic, 100 * z.taux, color=["#dc2626" if a.startswith("EMSR319") else "#2563eb" for a in z.index])
ax.set(xlabel="pic de pluie IMERG moyen de la zone (mm/h)", ylabel="segments inondés (%)", yscale="log",
       title="Gravité de la crue selon la pluie IMERG (rouge = Nabeul)")
plt.tight_layout(); plt.show()

mailles IMERG par zone : médiane 6, max 24
Spearman taux d'inondés / pic IMERG   : 0.52
Spearman taux d'inondés / total IMERG : 0.34


**Analyse — ce qu'apporte la pluie satellite IMERG.**

- **Les modèles qui utilisent la pluie progressent tous** :
  - V1 passe de 0,50 à 0,58 de ROC-AUC (+0,07) ;
  - V2 logistique de 0,61 à 0,69 (+0,08), et son PR-AUC de 0,165 à 0,233 ;
  - V2 gradient boosting de 0,69 à 0,74 (+0,05).

  Les deux témoins sans pluie (règle relative, V2 terrain) ne bougent pas : **l'amélioration vient bien de la pluie**.
- **Avec ERA5, la pluie dégradait le modèle** : V2 complet faisait moins bien que V2 terrain (0,69 contre 0,73). **Avec IMERG, elle aide** (0,74 contre 0,73).
  La qualité de la donnée pluie était donc le facteur limitant.
- **À Nabeul**, V2 complet passe :
  - de 0,74 à **0,80** de ROC-AUC, au niveau de la règle relative ;
  - de 26 % à **33 %** de rappel au niveau Élevé ;
  - de 7,8 % à **9,8 %** de précision au niveau Élevé, soit 16 fois le hasard.
- **IMERG explique en partie la gravité des crues.** La corrélation de rang entre le taux d'inondés d'une zone et son pic de pluie atteint **0,52**, alors qu'elle était nulle avec ERA5.
  Le cumul total explique moins bien (0,34) : **c'est l'intensité horaire qui déclenche les crues éclair**, plus que le volume.
- **Limite** : avec une maille de 10 km, IMERG ne distingue pas deux rues voisines. À l'intérieur d'une zone, le classement reste porté par le terrain.
- Ces chiffres comparent deux runs faits **avec les paramètres par défaut**, pour isoler l'effet de la pluie. Après réglage, V2 complet atteint 0,739 en moyenne et 0,81 à Nabeul.

## 7 ter. Synthèse : pluie seule, terrain seul, V2 complet

Pour savoir ce qu'apporte chaque type d'information, on compare V2 complet à deux références apprises de la même façon :
**pluie seule** (`hgb_rain`) et **terrain seul** (`hgb_terrain`).

Le **Brier Skill Score** (BSS) compare le Brier score du modèle à celui d'une prévision **constante** égale au taux d'inondés
des crues d'entraînement du pli (sans la crue testée) :

$$BSS = 1 - \frac{Brier_{modèle}}{Brier_{constante}}$$

BSS = 1 : parfait ; BSS = 0 : pas mieux que la constante ; BSS < 0 : **moins bien** que la constante.

In [16]:
syn_cols = ["roc_auc", "pr_auc", "recall_high", "false_alarm", "brier", "brier_ref", "bss"]
syn = main.groupby("model")[syn_cols].mean().loc[KINDS]
syn.index = [NAMES[k] for k in KINDS]
print("Moyenne des 13 crues (chaque crue jamais vue, délai 20 min, Élevé = 2 % de chaque zone) :")
display(syn.round(4))
syn_n = main[main.events == "EMSR319"].set_index("model")[syn_cols].loc[KINDS]
syn_n.index = [NAMES[k] for k in KINDS]
print("Nabeul 2018 :")
display(syn_n.round(4))
# BSS global sur toutes les lignes : référence = taux d'inondés des autres crues (celles du pli d'entraînement)
y_all = df[LABEL].to_numpy()
tot, cnt = df.groupby("event_id")[LABEL].sum(), df.groupby("event_id")[LABEL].size()
ref = df.event_id.map((tot.sum() - tot) / (cnt.sum() - cnt)).to_numpy()
pooled = pd.DataFrame({NAMES[k]: {"Brier": np.mean((oof[k] - y_all) ** 2), "Brier constante": np.mean((ref - y_all) ** 2)}
                       for k in KINDS}).T
pooled["BSS"] = 1 - pooled["Brier"] / pooled["Brier constante"]
print("Toutes les lignes ensemble :")
pooled.round(4)

Moyenne des 13 crues (chaque crue jamais vue, délai 20 min, Élevé = 2 % de chaque zone) :


,roc_auc,pr_auc,recall_high,false_alarm,brier,brier_ref,bss
V1 multi-criteria,0.5770,0.1441,0.0292,0.8273,0.1004,0.1008,0.0509
Relative susceptibility,0.7428,0.2235,0.0813,0.7672,0.0989,0.1008,-0.0072
V2 logistic,0.6937,0.2329,0.0725,0.7429,0.1059,0.1008,-0.0509
V2 gradient boosting,0.7387,0.2078,0.1013,0.7361,0.1016,0.1008,0.0170
V2 GB without drainage proxies,0.7320,0.2055,0.0987,0.7283,0.1022,0.1008,0.0104
V2 GB terrain only,0.7382,0.2420,0.1080,0.7179,0.1020,0.1008,-0.0207
V2 GB rain only,0.4987,0.1300,0.0265,0.8747,0.1032,0.1008,-0.0649


Nabeul 2018 :


,roc_auc,pr_auc,recall_high,false_alarm,brier,brier_ref,bss
V1 multi-criteria,0.5459,0.0069,0.0380,0.9886,0.0163,0.0201,0.1909
Relative susceptibility,0.8021,0.0301,0.1413,0.9575,0.0236,0.0201,-0.1747
V2 logistic,0.6953,0.0108,0.1196,0.9641,0.0203,0.0201,-0.0083
V2 gradient boosting,0.8134,0.0264,0.3533,0.8938,0.0189,0.0201,0.0584
V2 GB without drainage proxies,0.7994,0.0603,0.3315,0.9003,0.0197,0.0201,0.0184
V2 GB terrain only,0.7968,0.0287,0.2880,0.9134,0.0260,0.0201,-0.2954
V2 GB rain only,0.5430,0.0070,0.0000,1.0000,0.0379,0.0201,-0.8877


Toutes les lignes ensemble :


,Brier,Brier constante,BSS
V1 multi-criteria,0.1000,0.0989,-0.0116
Relative susceptibility,0.0977,0.0989,0.0118
V2 logistic,0.1176,0.0989,-0.1898
V2 gradient boosting,0.0996,0.0989,-0.0078
V2 GB without drainage proxies,0.1006,0.0989,-0.0179
V2 GB terrain only,0.1010,0.0989,-0.0215
V2 GB rain only,0.1030,0.0989,-0.0412


**Analyse — ce qu'apporte chaque type d'information** (moyenne des 13 crues, délai 20 min, Élevé = 2 % de chaque zone).

| Modèle | ROC-AUC | PR-AUC | Rappel Élevé | Fausses alertes | Brier | BSS |
|---|---|---|---|---|---|---|
| Pluie seule | 0,50 | 0,13 | 2,7 % | 87 % | 0,1032 | −0,065 |
| Terrain seul | 0,74 | **0,24** | **10,8 %** | **72 %** | 0,1020 | −0,021 |
| **V2 complet (terrain + pluie)** | **0,74** | 0,21 | 10,1 % | 74 % | 0,1016 | **+0,017** |
| V1 | 0,58 | 0,14 | 2,9 % | 83 % | 0,1004 | +0,051 |
| Prévision constante (taux d'entraînement) | 0,50 | – | – | – | 0,1008 | 0 |

**1. La pluie seule ne classe pas les routes à l'intérieur d'une zone, mais elle n'est pas inutile.**
- Son ROC-AUC est de 0,50, comme le hasard. Au niveau Élevé, elle trouve 2,7 % des inondations, avec 87 % de fausses alertes.
- Avec une maille de 10 km, toutes les routes d'un même quartier reçoivent la même pluie. La pluie ne peut donc pas dire *laquelle* va être inondée.
- Elle distingue en revanche les **zones** entre elles : le pic IMERG est corrélé au taux d'inondés de la zone (Spearman 0,52, section 7 bis). Elle est aussi la seule variable qui peut faire **évoluer le risque dans le temps**. Ce second rôle n'est pas testable ici, car l'inventaire ne contient qu'un orage par zone et aucun orage sans inondation.

**2. Le terrain fait l'essentiel du classement.**
- En moyenne, le terrain seul obtient le meilleur PR-AUC et le meilleur rappel.
- V2 complet fait mieux là où la pluie varie fortement à l'intérieur de la crue : Nabeul (rappel Élevé 35 % contre 29 %), SE Espagne et Valence 2024.
- V2 complet est aussi **le seul des deux à réagir à la prévision**. Le terrain seul donne la même carte qu'il pleuve 5 mm ou 150 mm : c'est une carte de susceptibilité, pas une alerte.

**3. Les probabilités ne battent pas une prévision constante.** C'est ce que montre le **Brier Skill Score (BSS)**.
- Le BSS de V2 complet est de **+0,017** en moyenne par crue, et de −0,008 sur toutes les lignes ensemble. En pratique, ses probabilités ne font **pas mieux qu'annoncer le même taux partout**.
- C'est la même chose pour tous les modèles (BSS entre −0,07 et +0,05). Le meilleur, V1 (+0,05), doit ce score au hasard : son score moyen tombe près du taux moyen.
- À Nabeul, le BSS de V2 est de +0,058, mais celui de V1 est plus élevé (+0,19), car V1 donne des scores très bas partout et Nabeul n'a que 0,6 % d'inondés.
- **Conclusion :** le modèle sait **classer** les routes (ROC-AUC 0,74, rappel 3,5 fois celui de V1), mais ses **probabilités absolues n'ont pas de valeur**.
  - C'est pourquoi les niveaux sont placés par rang et non par probabilité.
  - C'est aussi pourquoi le contrat (`docs/MODEL_CONTRACT.md`) demande d'afficher le niveau, et non le pourcentage.

## 8. Validation tunisienne : le pli Nabeul (EMSR319)

In [17]:
nab = res[(res.events == "EMSR319") & (res.target == "flooded") & (res.lead == MAIN_LEAD)].set_index("model").loc[KINDS, cols]
nab.index = [NAMES[k] for k in KINDS]
display(nab.round(3))
m = (df.event_id == "EMSR319").to_numpy()
y_n = df.loc[m, LABEL].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for k in KINDS:
    fpr, tpr, _ = roc_curve(y_n, oof[k][m]); axes[0].plot(fpr, tpr, color=COLORS[k], label=NAMES[k])
    pr, rc, _ = precision_recall_curve(y_n, oof[k][m]); axes[1].plot(rc, pr, color=COLORS[k], label=NAMES[k])
axes[0].plot([0, 1], [0, 1], "k:", lw=1); axes[0].set(xlabel="faux positifs", ylabel="rappel", title="ROC — Nabeul 2018")
axes[1].axhline(y_n.mean(), color="k", ls=":", lw=1, label="hasard")
axes[1].set(xlabel="rappel", ylabel="précision", title="Précision-rappel — Nabeul 2018", ylim=(0, max(0.1, 3 * y_n.mean() * 3)))
axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()

,roc_auc,pr_auc,recall_top1,recall_top2,recall_top5,recall_high,precision_high,accuracy_high
V1 multi-criteria,0.546,0.007,0.016,0.022,0.060,0.038,0.011,0.975
Relative susceptibility,0.802,0.030,0.071,0.114,0.348,0.141,0.042,0.976
V2 logistic,0.695,0.011,0.005,0.022,0.060,0.120,0.036,0.976
V2 gradient boosting,0.813,0.026,0.092,0.158,0.277,0.353,0.106,0.978
V2 GB without drainage proxies,0.799,0.060,0.207,0.250,0.342,0.332,0.100,0.978
V2 GB terrain only,0.797,0.029,0.049,0.087,0.217,0.288,0.087,0.978
V2 GB rain only,0.543,0.007,0.000,0.005,0.043,0.000,0.000,0.974


**Analyse du pli tunisien** (Nabeul 2018, jamais vu à l'entraînement, pluie IMERG, paramètres réglés).

| Modèle | ROC-AUC | Rappel top 5 % | Rappel Élevé (2 %) | Précision Élevé |
|---|---|---|---|---|
| V1 | 0,55 | 0,06 | 0,04 | 1,1 % |
| Règle relative | 0,80 | **0,35** | 0,14 | 4,2 % |
| V2 logistique | 0,70 | 0,06 | 0,12 | 3,6 % |
| **V2 gradient boosting** | **0,81** | 0,28 | **0,35** | **10,6 %** |
| V2 GB terrain seul | 0,80 | 0,22 | 0,29 | 8,7 % |

- **Sans aucune donnée tunisienne, V2 classe Nabeul mieux que toutes les autres approches (ROC-AUC 0,81)**, et bien mieux que V1 (0,55).
- **Au niveau d'alerte Élevé (2 % du réseau de chaque zone), V2 est nettement le meilleur.** Il détecte **35 % des segments inondés**, avec une précision de **10,6 %**, soit **18 fois le taux de base** (0,6 %).
  La règle relative en détecte 14 %, et V1 4 %.
- **À 5 % de budget sur tout Nabeul, la règle relative reste un peu devant** : 35 % contre 28 %. Le réglage a réduit l'écart, qui était de 35 % contre 20 % auparavant.
- **Attention : la règle relative a été choisie en regardant Nabeul** (voir `v1.relative_susceptibility`), et Nabeul a pesé pour 1/13 dans le réglage de V2. Ces deux scores sont donc un peu optimistes.

## 9. Métriques de la section 8

### 9.1 Niveaux d'alerte, rappel au niveau Élevé+ et fausses alertes

**Comment les niveaux sont placés.** Un seuil de probabilité unique ne se transfère pas d'une région à l'autre : le taux d'inondés va de 0,6 % (Nabeul)
à 27 % (Derna), et un seuil appris ailleurs mettait **16 % du réseau de Nabeul en alerte Élevé+, mais 0,1 % de celui de Valence**.
Les niveaux sont donc placés **par budget d'alerte dans chaque zone** (FR-24, `levels_by_share`) : dans chaque zone, les segments sont classés
par risque, et les **10 %** les plus risqués sont au moins *Modéré*, les **2 %** au moins *Élevé*, les **0,5 %** *Très élevé*.
Ce placement n'utilise aucune étiquette de la zone testée : il ne demande que les scores du modèle, disponibles en temps réel.

Mesures au niveau Élevé+ : **rappel** (part des inondations alertées), **précision** (part des alertes justes), **fausses alertes** (1 − précision)
et **accuracy** (part des segments bien classés, alerte ou non).

In [18]:
fa = main.pivot_table(index="events", columns="model", values=["recall_high", "precision_high", "alerts_per_1000"]).loc[events]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
w = 0.11
for i, k in enumerate(KINDS):
    axes[0].bar(np.arange(len(events)) + (i - 3) * w, fa[("recall_high", k)], w, color=COLORS[k], label=NAMES[k])
    axes[1].bar(np.arange(len(events)) + (i - 3) * w, fa[("precision_high", k)], w, color=COLORS[k])
axes[0].set(ylabel="rappel Élevé+", title="Rappel au niveau Élevé+ par crue (2 % du réseau en alerte)")
axes[1].plot(np.arange(len(events)), base.values, "k_", ms=14, mew=2, label="taux d'inondés (hasard)")
axes[1].set(ylabel="précision Élevé+", title="Précision des alertes Élevé+ par crue (trait noir = hasard)")
for ax in axes:
    ax.set_xticks(range(len(events)), events, rotation=45, fontsize=8)
axes[0].legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()
fa_mean = main.groupby("model")[["recall_high", "precision_high", "false_alarm", "accuracy_high", "alerts_per_1000"]].mean().loc[KINDS]
fa_mean.index = [NAMES[k] for k in KINDS]
fa_mean.loc["Référence : jamais d'alerte"] = [0, np.nan, np.nan, 1 - base.mean(), 0]
fa_mean.round(3)

,recall_high,precision_high,false_alarm,accuracy_high,alerts_per_1000
V1 multi-criteria,0.029,0.173,0.827,0.875,19.817
Relative susceptibility,0.081,0.233,0.767,0.878,19.817
V2 logistic,0.073,0.257,0.743,0.879,19.817
V2 gradient boosting,0.101,0.264,0.736,0.879,19.817
V2 GB without drainage proxies,0.099,0.272,0.728,0.879,19.817
V2 GB terrain only,0.108,0.282,0.718,0.880,19.817
V2 GB rain only,0.027,0.125,0.875,0.873,19.817
Référence : jamais d'alerte,0.000,NaN,NaN,0.888,0.000


**Analyse — niveaux par budget d'alerte** (2 % de chaque zone au niveau Élevé+). Moyenne sur les 13 crues jamais vues :

| Modèle | Rappel Élevé+ | Précision | Fausses alertes | Accuracy |
|---|---|---|---|---|
| V1 | 2,9 % | 17,3 % | 83 % | 87,5 % |
| Règle relative | 8,1 % | 23,3 % | 77 % | 87,8 % |
| V2 logistique | 7,3 % | 25,7 % | 74 % | 87,9 % |
| V2 GB | 10,1 % | 26,4 % | 74 % | 87,9 % |
| V2 GB terrain | **10,8 %** | **28,2 %** | **72 %** | **88,0 %** |
| V2 GB pluie seule | 2,7 % | 12,5 % | 87 % | 87,3 % |
| Jamais d'alerte | 0 % | – | – | 88,8 % |

- **Le volume d'alertes est maîtrisé** : exactement 2 % du réseau de chaque zone. Avec des seuils de probabilité appris ailleurs, on mettait en alerte 65 à 75 % du réseau.
- **Les alertes de V2 sont plus fiables que le hasard et que V1.**
  - La précision de V2 est de 26 à 28 %, contre environ 11 % pour le hasard (taux moyen d'inondés).
  - Une alerte Élevé de V2 est donc juste **2,4 fois plus souvent qu'une alerte au hasard**, et 1,5 fois plus souvent qu'une alerte de V1.
- **Le rappel reste faible (10–11 %).** Avec 2 % des routes en alerte, on ne peut couvrir qu'une petite part des zones inondées, qui représentent en moyenne 11 % du réseau cartographié.
- **L'accuracy (88 %) ne dépasse pas la référence « jamais d'alerte » (88,8 %).** Ce n'est pas un défaut du modèle.
  - Quand 89 % des segments sont secs, toute alerte dont la précision est inférieure à 50 % fait baisser l'accuracy.
  - C'est pourquoi la section 8 juge le système sur le **rappel** et les **fausses alertes**, pas sur l'accuracy.
- **Les crues très étendues profitent moins de ce budget fixe.**
  - Derna, Valence 2024 et SE Espagne ont 11 à 27 % d'inondés, et 2 % d'alertes y couvrent mécaniquement peu de segments inondés.
  - À l'inverse, les crues rares en profitent : à Nabeul (0,6 % d'inondés) et en Sicile 2021, V2 détecte **32 à 35 %** des segments inondés.

**Plafond du rappel au budget de 2 %.** Dans une zone de *n* segments dont *f* sont inondés, le niveau Élevé+ ne contient que
⌊0,02 · n⌋ segments, donc même un modèle parfait ne peut alerter que min(*f*, ⌊0,02 · n⌋) segments inondés.
On vérifie ce calcul de deux façons : par la formule, et en passant les **vraies étiquettes** (modèle « oracle ») dans la même
fonction `levels_by_share` que les modèles.

In [19]:
rng_o = np.random.default_rng(0)
rows = []
for e in events:
    m_e = (df.event_id == e).to_numpy()
    y_e, a_e = y_all[m_e], df["aoi"].to_numpy()[m_e]
    z = pd.DataFrame({"y": y_e, "aoi": a_e}).groupby("aoi").agg(n=("y", "size"), f=("y", "sum"))
    formula = np.minimum(z.f, np.floor(0.02 * z.n)).sum() / z.f.sum()
    oracle_lv = v2.levels_by_share(y_e + 1e-6 * rng_o.random(len(y_e)), a_e)   # vraies étiquettes, égalités au hasard
    oracle = (np.isin(oracle_lv, ["High", "Very high"]) & (y_e == 1)).sum() / y_e.sum()
    v2_rec = main[(main.events == e) & (main.model == "hgb")].recall_high.iloc[0]
    rows.append({"crue": e, "taux d'inondés": y_e.mean(), "plafond (formule)": formula, "plafond (oracle)": oracle,
                 "rappel V2": v2_rec, "V2 / plafond": v2_rec / formula})
ceiling = pd.DataFrame(rows).set_index("crue")
print(f"plafond moyen : formule {ceiling['plafond (formule)'].mean():.3f}, oracle {ceiling['plafond (oracle)'].mean():.3f} | "
      f"V2 {ceiling['rappel V2'].mean():.3f}, soit {ceiling['V2 / plafond'].mean():.0%} du plafond en moyenne")
ceiling.round(3)

plafond moyen : formule 0.385, oracle 0.385 | V2 0.101, soit 30% du plafond en moyenne


,taux d'inondés,plafond (formule),plafond (oracle),rappel V2,V2 / plafond
crue,,,,,
EMSR257,0.173,0.114,0.114,0.000,0.000
EMSR319,0.006,1.000,1.000,0.353,0.353
EMSR323,0.220,0.087,0.087,0.032,0.368
EMSR324,0.032,0.628,0.628,0.217,0.345
EMSR327,0.023,0.862,0.862,0.069,0.080
EMSR388,0.107,0.187,0.187,0.033,0.176
EMSR413,0.178,0.111,0.111,0.081,0.729
EMSR480,0.138,0.142,0.142,0.076,0.531
EMSR548,0.006,0.906,0.906,0.315,0.348


**Analyse — vérification du plafond.**

La formule et le modèle oracle donnent **exactement le même plafond** pour les 13 crues. La valeur moyenne de **38,5 %** est donc vérifiée.

Ce plafond est le meilleur rappel possible au niveau Élevé avec un budget de 2 % par zone. Il dépend fortement du taux d'inondés de chaque crue :

| Crue | Taux d'inondés | Plafond | Rappel V2 | Part du plafond |
|---|---|---|---|---|
| Nabeul 2018 | 0,6 % | 100 % | 35 % | 35 % |
| Sicile 2021 | 0,6 % | 91 % | 32 % | 35 % |
| Derna 2023 | 27 % | 7 % | 3 % | 36 % |
| Valence 2024 | 20 % | 10 % | 5 % | 48 % |
| Attique ouest 2019 | 18 % | 11 % | 8 % | 73 % |
| Mandra 2017 | 17 % | 11 % | 0 % | 0 % |

- **V2 atteint en moyenne 30 % du plafond** (10,1 % de rappel sur 38,5 % possibles). C'est très au-dessus du hasard, qui atteindrait 2 % de rappel, mais encore loin d'un modèle parfait.
- **Le rappel moyen de 10 % ne veut donc pas dire que le modèle « rate 90 % des inondations ».**
  - Sur les crues très étendues (Derna, Valence, Mallorca), même un modèle parfait ne pourrait en signaler que 7 à 10 % avec 2 % d'alertes.
  - Le vrai levier pour augmenter le rappel est le **budget d'alerte**, qui est un choix de la protection civile (courbe ci-dessous).
- **Échecs nets** : Mandra (0 % du plafond), Andalousie et Sicile 2023 (8 %), Valence 2022 (9 %).

**Choisir le budget.** Le budget est un choix de la protection civile : plus on alerte de routes, plus on couvre d'inondations, mais plus il y a de fausses alertes.
La courbe ci-dessous montre ce compromis pour chaque budget (le même dans chaque zone), à partir des prédictions sur les crues jamais vues.

In [20]:
area, y = df["aoi"].to_numpy(), df[LABEL].to_numpy()
rows = []
for k in ["v1", "v1_relative", "hgb", "hgb_terrain"]:
    rank = pd.Series(oof[k]).groupby(area).rank(pct=True, ascending=False, method="first").to_numpy()
    for share in [0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30]:
        alert = rank <= share
        for e in events:
            m_e = (df.event_id == e).to_numpy()
            a_e, y_e = alert[m_e], y[m_e]
            tp = (a_e & (y_e == 1)).sum()
            rows.append({"model": k, "budget": share, "event": e, "recall": tp / max(y_e.sum(), 1),
                         "precision": tp / max(a_e.sum(), 1), "accuracy": (a_e == (y_e == 1)).mean()})
curve = pd.DataFrame(rows).groupby(["model", "budget"])[["recall", "precision", "accuracy"]].mean()
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for k in ["v1", "v1_relative", "hgb", "hgb_terrain"]:
    for ax, c in zip(axes, ["recall", "precision", "accuracy"]):
        ax.plot(100 * curve.loc[k].index, curve.loc[k, c], marker="o", color=COLORS[k], label=NAMES[k])
axes[0].plot([0, 30], [0, 0.3], "k:", lw=1, label="hasard")
axes[1].axhline(base.mean(), color="k", ls=":", lw=1)
axes[2].axhline(1 - base.mean(), color="k", ls=":", lw=1, label="jamais d'alerte")
for ax, c in zip(axes, ["rappel", "précision", "accuracy"]):
    ax.set(xlabel="budget : % des segments de chaque zone en alerte", ylabel=c, title=f"{c} selon le budget")
axes[0].legend(fontsize=7); axes[2].legend(fontsize=7); plt.tight_layout(); plt.show()
curve.loc["hgb_terrain"].round(3)

,recall,precision,accuracy
budget,,,
0.005,0.040,0.297,0.886
0.010,0.070,0.298,0.884
0.020,0.108,0.282,0.880
0.050,0.220,0.259,0.864
0.100,0.335,0.228,0.834
0.200,0.507,0.199,0.768
0.300,0.619,0.177,0.695


**Analyse — choix du budget** (courbe calculée pour V2 terrain, le modèle au meilleur rappel moyen).

| Budget par zone | Rappel | Précision | Accuracy |
|---|---|---|---|
| 0,5 % | 4 % | 30 % | 88,6 % |
| 1 % | 7 % | 30 % | 88,4 % |
| 2 % | 11 % | 28 % | 88,0 % |
| 5 % | 22 % | 26 % | 86,4 % |
| 10 % | 34 % | 23 % | 83,4 % |
| 20 % | 51 % | 20 % | 76,8 % |
| 30 % | 62 % | 18 % | 69,5 % |

- **Jusqu'à 10 % de budget, la précision reste stable** (23 à 30 %), puis elle baisse doucement. Le rappel monte presque linéairement.
- **La cible de 70 % de rappel de la section 8 n'est pas réaliste avec ces données.** Il faudrait alerter plus de 30 % du réseau, avec plus de 80 % de fausses alertes.
- **Proposition : Élevé = 2 % et Modéré = 10 %** de chaque zone (`schema.ALERT_SHARE`).
  - Le niveau Élevé reste rare et environ 2,5 fois plus fiable que le hasard.
  - Le niveau Modéré couvre environ un tiers des inondations.
  - Ces budgets sont un réglage de la protection civile : on peut les changer sans réentraîner le modèle.

### 9.2 Recall à budget d'alerte fixe

En pratique, on ne peut alerter qu'une petite part du réseau. Le rappel quand on n'alerte que les 1 %, 2 % ou 5 % de segments
les plus risqués est la mesure la plus parlante pour la protection civile, et elle ne dépend pas du seuil.

In [21]:
budget = main.groupby("model")[["recall_top1", "recall_top2", "recall_top5"]].mean().loc[KINDS]
ax = budget.rename(index=NAMES).T.plot.bar(figsize=(9, 4), color=[COLORS[k] for k in KINDS], rot=0)
for share, x in zip((0.01, 0.02, 0.05), range(3)):
    ax.plot([x - 0.4, x + 0.4], [share, share], "k:", lw=1)
ax.set(ylabel="part des inondations couvertes", title="Rappel à budget fixe (pointillé = hasard)")
ax.set_xticklabels(["top 1 %", "top 2 %", "top 5 %"]); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
budget.rename(index=NAMES).round(3)

,recall_top1,recall_top2,recall_top5
model,,,
V1 multi-criteria,0.015,0.027,0.067
Relative susceptibility,0.050,0.078,0.198
V2 logistic,0.037,0.072,0.160
V2 gradient boosting,0.041,0.084,0.187
V2 GB without drainage proxies,0.053,0.089,0.185
V2 GB terrain only,0.060,0.100,0.216
V2 GB rain only,0.012,0.029,0.063


**Analyse — rappel à budget fixe** (classement sur toute la crue, sans découpage par zone).

- En alertant les **5 % de segments les plus risqués**, on couvre :
  - **22 %** des inondations avec V2 terrain ;
  - **20 %** avec la règle relative ;
  - **19 %** avec V2 complet ;
  - **6,3 %** avec la pluie seule ;
  - **6,7 %** avec V1, à peine mieux que le hasard (5 %).
- Ce classement global favorise les modèles fondés sur le terrain. V2 complet donne plus de poids à la pluie, qui sépare surtout les zones entre elles, alors que ce test classe tous les segments de la crue ensemble.
- **Pour la protection civile**, cela veut dire que V2 sait indiquer quels secteurs surveiller en priorité, mais pas encore désigner précisément les quelques routes à fermer.

### 9.3 Délai d'anticipation (lead time)

La pluie au moment du passage est remplacée par une **prévision** dont l'erreur croît avec le délai (0 → 30 min),
plus grande quand la pluie 15 min est interpolée d'une prévision horaire.

In [22]:
lt = res[res.target == "flooded"].groupby(["model", "lead"])[["pr_auc", "roc_auc", "recall_top2"]].mean()
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for k in KINDS:
    for ax, c in zip(axes, ["roc_auc", "pr_auc", "recall_top2"]):
        ax.plot(LEADS_MIN, lt.loc[k, c], marker="o", color=COLORS[k], label=NAMES[k])
for ax, c in zip(axes, ["ROC-AUC", "PR-AUC", "rappel top 2 %"]):
    ax.set(xlabel="délai de prévision (min)", ylabel=c, title=f"{c} selon le délai")
axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()
lt["pr_auc"].unstack().loc[KINDS].rename(index=NAMES).round(4)

lead,0,10,20,30
model,,,,
V1 multi-criteria,0.1550,0.1516,0.1441,0.1417
Relative susceptibility,0.2235,0.2235,0.2235,0.2235
V2 logistic,0.2381,0.2395,0.2329,0.2263
V2 gradient boosting,0.2119,0.2115,0.2078,0.2044
V2 GB without drainage proxies,0.2085,0.2087,0.2055,0.2031
V2 GB terrain only,0.2420,0.2420,0.2420,0.2420
V2 GB rain only,0.1429,0.1418,0.1300,0.1258


**Analyse — délai d'anticipation.** Le PR-AUC baisse quand la pluie prévue devient moins précise :

| Modèle | PR-AUC à 0 min | PR-AUC à 30 min | Perte |
|---|---|---|---|
| V2 gradient boosting | 0,212 | 0,204 | ≈ 3,5 % |
| V2 logistique | 0,238 | 0,226 | ≈ 5 % |
| V1 | 0,155 | 0,142 | ≈ 9 % |
| V2 pluie seule | 0,143 | 0,126 | ≈ 12 % |

- **V2 gradient boosting est le plus robuste.** Ses contraintes monotones et la part du terrain dans ses décisions le rendent peu sensible à l'erreur de prévision. **Il reste utilisable 30 min à l'avance.**
- **La pluie seule perd le plus (12 %)**, puis V1 (9 %), qui donne un poids fixe de 30 % à la pluie. Plus un modèle dépend de la prévision, plus l'erreur de prévision le pénalise.
- **Les modèles sans pluie** (V2 terrain, règle relative) ne bougent pas.
- **Limite** : l'erreur de prévision est **simulée** (bruit log-normal croissant avec le délai). La vraie erreur d'un modèle de prévision (AROME, ECMWF) sur des orages convectifs sera probablement plus forte.

### 9.4 Calibration comparée à V1

Une probabilité de 10 % doit correspondre à 10 % de segments inondés. On compare les probabilités hors-pli (délai 20 min)
à la fréquence observée, avec le **Brier score** et l'**ECE** (erreur de calibration attendue, plus petit = mieux).

In [23]:
y = df[LABEL].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
rows = []
for k in KINDS:
    p = oof[k]
    edges = np.unique(np.quantile(p, np.linspace(0, 1, 13)))
    idx = np.clip(np.searchsorted(edges, p, side="right") - 1, 0, len(edges) - 2)
    pts = np.array([(p[idx == b].mean(), y[idx == b].mean()) for b in range(len(edges) - 1) if (idx == b).any()])
    for ax in axes:
        ax.plot(*pts.T, marker="o", color=COLORS[k], label=NAMES[k])
    rows.append({"modèle": NAMES[k], "proba moyenne": p.mean(), "fréquence réelle": y.mean(),
                 "Brier": np.mean((p - y) ** 2), "ECE": ece(p, y)})
for ax, lim in zip(axes, [1, 0.3]):
    ax.plot([0, lim], [0, lim], "k--", lw=1); ax.set(xlim=(0, lim), ylim=(0, lim), xlabel="probabilité prédite", ylabel="fréquence observée")
axes[0].set_title("Diagramme de fiabilité (toutes crues)"); axes[1].set_title("Zoom 0–30 %")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(OUT / "reliability.png", dpi=130); plt.show()
calib = pd.DataFrame(rows).set_index("modèle"); calib.round(4)

,proba moyenne,fréquence réelle,Brier,ECE
modèle,,,,
V1 multi-criteria,0.1234,0.108,0.1000,0.0549
Relative susceptibility,0.1129,0.108,0.0977,0.0422
V2 logistic,0.1558,0.108,0.1176,0.1428
V2 gradient boosting,0.1078,0.108,0.0996,0.0949
V2 GB without drainage proxies,0.1105,0.108,0.1006,0.0954
V2 GB terrain only,0.1069,0.108,0.1010,0.0789
V2 GB rain only,0.1256,0.108,0.1030,0.1071


In [24]:
# Calibration crue par crue : moyenne prédite contre taux réel de la crue testée
cal_ev = pd.DataFrame({k: pd.Series(oof[k]).groupby(df.event_id.values).mean() for k in KINDS}).loc[events]
cal_ev.insert(0, "taux réel", base.values)
cal_ev.rename(columns=NAMES).round(3)

,taux réel,V1 multi-criteria,Relative susceptibility,V2 logistic,V2 gradient boosting,V2 GB without drainage proxies,V2 GB terrain only,V2 GB rain only
EMSR257,0.173,0.151,0.111,0.107,0.102,0.111,0.118,0.102
EMSR319,0.006,0.101,0.131,0.126,0.119,0.123,0.141,0.168
EMSR323,0.220,0.088,0.111,0.106,0.097,0.094,0.098,0.099
EMSR324,0.032,0.117,0.114,0.111,0.102,0.101,0.110,0.103
EMSR327,0.023,0.051,0.113,0.109,0.098,0.098,0.118,0.101
EMSR388,0.107,0.209,0.110,0.281,0.138,0.149,0.110,0.170
EMSR413,0.178,0.064,0.110,0.103,0.096,0.095,0.100,0.098
EMSR480,0.138,0.107,0.112,0.107,0.103,0.104,0.101,0.113
EMSR548,0.006,0.094,0.134,0.125,0.118,0.117,0.124,0.123
EMSR569,0.067,0.093,0.114,0.107,0.100,0.099,0.107,0.101


**Analyse — calibration comparée à V1.**

- **Calibration globale de V2 gradient boosting.**
  - Sa probabilité moyenne (0,108) correspond exactement au taux réel (0,108).
  - Son Brier score (0,0996) est légèrement meilleur que celui de V1 (0,1000).
- **L'ECE est en revanche moins bonne que celle de V1** (0,095 contre 0,055) : **la section 8 n'est donc que partiellement satisfaite**.
  - Avec IMERG, V1 est devenu mieux calibré par hasard : son score moyen est proche du taux moyen.
  - La régression logistique est mal calibrée (ECE 0,143) : elle surestime fortement la crue SE Espagne (0,28 prédit contre 0,11 réel).
- **Le tableau par crue montre la vraie limite.** V2 prédit **0,06–0,14 pour toutes les crues**, que le taux réel soit de 0,6 % (Nabeul, Sicile 2021) ou de 27 % (Derna).
  - Le modèle sait classer les segments *à l'intérieur* d'une crue, mais pas encore dire *combien* une crue va inonder.
  - Le taux d'inondés dépend largement du découpage des cartes Copernicus, et aucune variable ne peut l'expliquer.
- **Pour la Tunisie**, les probabilités de V2 sont **à recalibrer localement** dès qu'un deuxième événement tunisien sera disponible.
  - En attendant, il faut les présenter comme un **classement relatif**, ce que font les niveaux par budget, et non comme une probabilité absolue.
- **Brier Skill Score** (section 7 ter) : il est proche de 0 pour tous les modèles (V2 +0,017 en moyenne par crue). Les probabilités ne battent pas une prévision constante égale au taux d'entraînement, ce qui confirme qu'elles ne doivent pas être affichées comme des chances réelles.

### 9.5 Utilité du facteur drainage

Deux tests : (1) **ablation** — le gradient boosting avec et sans les variables drainage (`drain_inlet_density`, `dry_spell_days`,
`osm_completeness`, `basin_reports_prior`, et les variables dérivées) ; (2) **test de Mann-Whitney** — parmi les segments inondés,
ceux à forte vulnérabilité drainage sont-ils inondés sous une pluie plus faible ?

In [25]:
abl = main.pivot_table(index="events", columns="model", values=["pr_auc", "roc_auc"]).loc[events]
d = pd.DataFrame({"ROC avec drainage": abl[("roc_auc", "hgb")], "ROC sans drainage": abl[("roc_auc", "hgb_no_drainage")],
                  "PR avec drainage": abl[("pr_auc", "hgb")], "PR sans drainage": abl[("pr_auc", "hgb_no_drainage")]})
d["écart PR"] = d["PR avec drainage"] - d["PR sans drainage"]
display(d.round(4))
print("écart moyen de PR-AUC (avec − sans) :", round(d["écart PR"].mean(), 4),
      "| crues où le drainage aide :", int((d["écart PR"] > 0).sum()), "/", len(d))
print("\nvariables drainage disponibles dans l'inventaire :")
print(inv[["drain_inlet_density", "dry_spell_days", "osm_completeness"]].agg(["nunique", "min", "max"]).T)
flooded = inv[inv[LABEL] == 1]
dv = drainage_vulnerability(flooded)
hi, lo = flooded[dv >= dv.quantile(2 / 3)], flooded[dv <= dv.quantile(1 / 3)]
t = mannwhitneyu(hi.rain_eta_mmh, lo.rain_eta_mmh, alternative="less")
print(f"\nMann-Whitney : pluie médiane à l'inondation, forte vulnérabilité {hi.rain_eta_mmh.median():.1f} mm/h "
      f"contre faible {lo.rain_eta_mmh.median():.1f} mm/h, p = {t.pvalue:.3g}")

,ROC avec drainage,ROC sans drainage,PR avec drainage,PR sans drainage,écart PR
events,,,,,
EMSR257,0.3198,0.3144,0.1193,0.1185,0.0008
EMSR319,0.8134,0.7994,0.0264,0.0603,-0.0339
EMSR323,0.5913,0.6051,0.2555,0.2648,-0.0093
EMSR324,0.9103,0.9051,0.2654,0.2490,0.0164
EMSR327,0.8617,0.8493,0.0957,0.0941,0.0016
EMSR388,0.7675,0.7621,0.2342,0.2290,0.0052
EMSR413,0.7248,0.7282,0.4260,0.4303,-0.0043
EMSR480,0.7949,0.7796,0.2867,0.2710,0.0157
EMSR548,0.9342,0.9315,0.0727,0.0573,0.0154


écart moyen de PR-AUC (avec − sans) : 0.0023 | crues où le drainage aide : 9 / 13

variables drainage disponibles dans l'inventaire :
                     nunique  min   max
drain_inlet_density     38.0  0.0  60.0
dry_spell_days           8.0  1.0  60.0
osm_completeness         1.0  0.5   0.5

Mann-Whitney : pluie médiane à l'inondation, forte vulnérabilité 15.9 mm/h contre faible 18.1 mm/h, p = 4.14e-53


**Analyse — utilité du facteur drainage (section 8).**

- **Ablation.** Garder les variables drainage améliore très légèrement le PR-AUC moyen (+0,002). Elles aident sur **9 crues sur 13**.
- **À Nabeul, le résultat est mitigé.**
  - Avec le drainage, le ROC-AUC est meilleur (0,813 contre 0,799), de même que le rappel au niveau Élevé (35 % contre 33 %).
  - Sans le drainage, le PR-AUC est meilleur (0,060 contre 0,026), de même que le rappel top 5 % (34 % contre 28 %).
  - Le drainage aide donc à placer les segments au niveau d'une zone, mais pas à trouver les tout premiers points noirs.
- **L'effet est faible, et ces variables ne mesurent pas vraiment le drainage ici.**
  - Les avaloirs OSM sont rarissimes, et `osm_completeness` est constante.
  - Le gain vient en pratique des **jours secs avant l'orage** (`dry_spell_days`) et du **béton**, qui entrent dans `drainage_vuln`.
  - Un sol sec et compacté ruisselle davantage : c'est un effet d'**état du sol**, pas d'avaloirs bouchés.
- **Test de Mann-Whitney.** Les segments inondés à forte vulnérabilité l'ont été sous une pluie plus faible : 15,9 contre 18,1 mm/h, p ≈ 10⁻⁵³.
  - Ce résultat va dans le sens de l'hypothèse FR-09.
  - Mais il est **confondu** avec le béton et la zone : il ne prouve pas l'effet des avaloirs.
- **Conclusion : tel qu'il est mesuré, le facteur drainage n'est pas démontré.**
  - Pour conclure, il faut des données d'avaloirs réelles, comme le cadastre de l'ONAS ou les signalements « avaloir bouché » de l'application (FR-21).
  - À New York, où ces signalements existaient, ils aidaient le modèle, mais pas la densité d'avaloirs.

## 10. Modèle final et importance des variables

In [26]:
rng = np.random.default_rng(SEED + 1)
train_all = with_forecast_error(df, rng.uniform(0, 30, len(df)), rng)
finals = {k: v2.fit(k, train_all, SEED, neg_sample=NEG_SAMPLE, levels=LEVELS_MODE, params=PARAMS) for k in ["hgb", "hgb_terrain", "logistic"]}
for k, b in finals.items():
    joblib.dump(b, MODELS / f"v2_{k}.joblib")
pd.DataFrame({NAMES[k]: b.thresholds for k, b in finals.items()}).T.round(4)

,Moderate,High,Very high
V2 gradient boosting,0.1231,0.2271,0.3454
V2 GB terrain only,0.1368,0.1368,0.1368
V2 logistic,0.1070,0.1173,0.1427


In [27]:
BEST = "hgb"   # modèle complet : terrain + pluie IMERG
final = finals[BEST]
X = v2.features(BEST, df)[final.columns]
sample = np.random.default_rng(0).choice(len(X), 20000, replace=False)
imp = permutation_importance(final.model, X.iloc[sample], y[sample], scoring="average_precision", n_repeats=5, random_state=0)
order = np.argsort(imp.importances_mean)[-15:]
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.barh(np.array(final.columns)[order], imp.importances_mean[order], xerr=imp.importances_std[order], color="#dc2626")
ax.set(xlabel="baisse de PR-AUC quand la variable est mélangée", title=f"Importance des variables — {NAMES[BEST]}")
plt.tight_layout(); plt.savefig(OUT / "importance.png", dpi=130); plt.show()

**Analyse — importance des variables.** L'importance est calculée pour le modèle final (V2 gradient boosting réglé, avec le terrain et la pluie IMERG), sur un échantillon de l'entraînement. Elle est donc optimiste.

1. **`rain_24h_mm`** (pluie des 24 h avant le pic) est de loin la variable la plus importante. Elle mesure la **saturation du sol** : une crue éclair survient quand une forte averse tombe sur un sol déjà mouillé.
2. **L'altitude relative** passe deuxième, devant le **béton** (`impervious_pct`, `impervious_frac`). Avec `max_features` = 0,5, le modèle réglé dépend moins du béton que le modèle par défaut.
3. Viennent ensuite **les jours secs avant l'orage**, le **percentile d'altitude**, la **distance à l'oued**, les **cumuls sur 1 h et 3 h** et la **profondeur de cuvette**.
4. Le pic instantané (`rain_eta_mmh`), la vulnérabilité drainage et la pente apportent peu une fois les autres variables connues.

Le modèle combine donc ce qu'attend la physique : **pluie accumulée sur la maille × position basse proche de l'oued**.

Le béton garde un poids en partie **régional** : les zones espagnoles cartographiées entourent des villes très bétonnées, alors qu'à Nabeul l'eau a touché surtout la plaine agricole. C'est le point à surveiller lors de la recalibration tunisienne.

## 11. Conclusion

### Ce que montre ce notebook

1. **Un pipeline V2 complet sur données réelles** :
   - inventaire satellite Copernicus EMS (13 crues, 28 zones, 6 pays) ;
   - pluie satellite NASA GPM IMERG (10 km, 30 min) ;
   - contrôle qualité et variables physiques ;
   - réglage des hyperparamètres et validation *leave-one-event-out* ;
   - calibration et niveaux d'alerte par budget ;
   - délai d'anticipation et ablation du drainage.

   Le modèle final est enregistré dans `models/med_imerg_tuned/`. Le contrat d'utilisation est dans `docs/MODEL_CONTRACT.md`.
2. **V2 dépasse nettement V1 sur des crues jamais vues** :
   - ROC-AUC 0,74 contre 0,58 ;
   - précision des alertes Élevé de 26 % contre 17 %, pour un hasard à environ 11 % ;
   - il reste utilisable 30 min à l'avance (−3,5 % de PR-AUC).
3. **Le modèle se transfère à la Tunisie.** Appris sans aucune donnée tunisienne, V2 classe Nabeul 2018 avec une ROC-AUC de **0,81**, la meilleure de tous les modèles.
   Au niveau Élevé (2 % du réseau), il détecte **35 % des segments inondés**, avec une précision **18 fois supérieure au hasard**.
4. **La pluie satellite était le facteur limitant.** Remplacer ERA5 par IMERG fait gagner +0,05 à +0,08 de ROC-AUC aux modèles qui utilisent la pluie.
   Le réglage des hyperparamètres n'ajoute qu'un petit gain (+0,003 en moyenne, +0,015 à Nabeul).
5. **V2 égale la meilleure règle simple en moyenne** (0,74 contre 0,74). Il la dépasse au niveau d'alerte Élevé à Nabeul : 35 % contre 14 % de rappel.
6. **Le terrain fait le classement, la pluie le rend dynamique.**
   - La pluie seule ne classe pas les routes (ROC-AUC 0,50).
   - Associée au terrain, elle améliore les crues étendues et permet au risque de changer avec la prévision.
7. **Rappel replacé dans son plafond** : V2 atteint 30 % du meilleur rappel possible au budget de 2 % (10,1 % sur 38,5 %).

### Limites

- **13 crues seulement, dont une seule tunisienne.** Chaque chiffre par crue repose sur un seul événement.
- **Étiquettes satellites.** Elles montrent l'eau quelques jours après le pic, avec des manques possibles (nuages, arbres, rues étroites), et non une route coupée à une heure précise. Le taux d'inondés dépend aussi du découpage des zones.
- **Pluie à 10 km.** IMERG sous-estime encore les cellules orageuses : 78 mm à Nabeul contre plus de 200 mm mesurés.
- **Calibration.** Les probabilités ne battent pas une prévision constante (BSS ≈ 0) et ne suivent pas la gravité propre à chaque crue. Il faut recalibrer localement.
- **Crues torrentielles urbaines** (Mandra, Derna) : elles ne sont pas prévues.
- **Facteur drainage** : non démontré sans vraies données d'avaloirs.
- **Rappel.** À un budget raisonnable (2 % en Élevé), il est d'environ 10 % en moyenne. La cible de 70 % n'est pas atteignable sans alerter un tiers du réseau.
- **Erreur de prévision simulée**, et non mesurée sur de vraies prévisions météo.

### Ce que le modèle attend pour progresser

1. **Plus de crues tunisiennes cartographiées**, à fournir par la partie données du projet. Le pipeline réentraîne et recalibre automatiquement (`python -m flood_risk.train --levels area --hgb-params models/med_imerg/hgb_params.json`).
2. **Une pluie plus fine** : radar ou pluviomètres de l'INM.
3. **Les signalements de l'application (FR-21)**, pour constituer un historique local et tester vraiment le drainage.